# Case Study: Cleaning a DeFi Airdrop from Sybil Farmers

### Behavioral clustering and evidence-driven investigation using onchain-sybil-detector

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.decomposition import PCA
from sklearn.metrics import precision_score, recall_score, f1_score

from sybil_detector import (
    SybilDetector,
    extract_features,
    generate_adversarial_sybils,
    generate_synthetic_sybil_network,
)
from sybil_detector.airdrop_hunter_osd import run_airdrop_hunter
from sybil_detector.chains_osd import SUPPORTED_CHAINS
from sybil_detector.explainer_osd import explain_wallet_linkage

pio.templates.default = 'plotly_dark'
pd.set_option('display.max_colwidth', 120)
SEED = 42

## 1) The Problem

Airdrops can move millions of dollars in a single distribution window. Farmers create coordinated wallet clusters to capture more than their fair share. Enterprise tooling is expensive, so we use this open-source pipeline to quantify abuse before tokens are sent.

In [2]:
problem_df = pd.DataFrame(
    {
        'segment': ['Legitimate participants', 'Suspected Sybil', 'Confirmed Sybil'],
        'share': [65, 27, 8],
        'color': ['#2ca02c', '#ff7f0e', '#d62728'],
    }
)

fig1 = px.pie(
    problem_df,
    names='segment',
    values='share',
    title='Chart 1. Typical Airdrop Participant Mix',
    color='segment',
    color_discrete_map={
        'Legitimate participants': '#2ca02c',
        'Suspected Sybil': '#ff7f0e',
        'Confirmed Sybil': '#d62728',
    },
    template='plotly_dark',
)
fig1.update_traces(textposition='inside', textinfo='percent+label')
fig1.add_annotation(
    text='Up to 40% of airdrop tokens go to Sybils (Arbitrum data)',
    x=0.5,
    y=-0.1,
    xref='paper',
    yref='paper',
    showarrow=False,
    font=dict(size=12),
)
fig1.show()

## 2) Scanning the Campaign

We simulate a campaign with 700 participant wallets and run the full detection pipeline. The output gives per-wallet Sybil probability, cluster IDs, and coordination evidence. We use those outputs to prioritize wallets for review before an airdrop snapshot.

In [3]:
tx_raw, labels = generate_synthetic_sybil_network(
    num_legit=550,
    num_clusters=10,
    wallets_per_cluster=15,
    seed=SEED,
)

participants = sorted(labels['address'].astype(str).str.lower().unique().tolist())
tx = tx_raw[tx_raw['address'].astype(str).str.lower().isin(participants)].copy()

features = extract_features(tx)
detector = SybilDetector(min_cluster_size=3, min_samples=2)
results = detector.fit_predict(features)

analysis_df = results.merge(labels[['address', 'is_sybil', 'cluster_id']].rename(columns={'cluster_id': 'label_cluster_id'}), on='address', how='left')
analysis_df['predicted_sybil'] = (analysis_df['sybil_probability'] >= 0.5).astype(int)
analysis_df['cluster_group'] = np.where(analysis_df['cluster_id'] >= 0, 'Sybil cluster', 'Benign/noise')

wallet_count = int(analysis_df['address'].nunique())
tx_count = int(tx['tx_hash'].nunique())
cluster_count = int(analysis_df.loc[analysis_df['cluster_id'] >= 0, 'cluster_id'].nunique())

print(f'We scanned {wallet_count} wallets across {tx_count} transactions.')
print(f'Detected clusters: {cluster_count}')

We scanned 700 wallets across 14102 transactions.
Detected clusters: 10


In [4]:
numeric_cols = [
    c for c in features.columns
    if c not in {'address', 'common_funder_address', 'hour_histogram'} and pd.api.types.is_numeric_dtype(features[c])
]

pca = PCA(n_components=2, random_state=SEED)
components = pca.fit_transform(features[numeric_cols].fillna(0.0))
plot_df = analysis_df.copy()
plot_df['pc1'] = components[:, 0]
plot_df['pc2'] = components[:, 1]

unique_clusters = sorted([int(c) for c in plot_df['cluster_id'].unique() if int(c) >= 0])
reds = px.colors.sequential.Reds
cluster_color_map = {'noise': '#8c8c8c'}
for i, cid in enumerate(unique_clusters):
    cluster_color_map[f'cluster_{cid}'] = reds[min(len(reds) - 1, 2 + (i % (len(reds) - 2)))]

plot_df['cluster_label'] = plot_df['cluster_id'].apply(lambda c: 'noise' if int(c) < 0 else f'cluster_{int(c)}')

fig2 = px.scatter(
    plot_df,
    x='pc1',
    y='pc2',
    color='cluster_label',
    size='sybil_probability',
    hover_data=['address', 'cluster_id', 'sybil_probability'],
    title='Chart 2. Cluster Map (PCA Projection of Behavioral Features)',
    labels={
        'pc1': 'Principal Component 1',
        'pc2': 'Principal Component 2',
        'cluster_label': 'Detected cluster',
        'sybil_probability': 'Sybil probability',
    },
    color_discrete_map=cluster_color_map,
    template='plotly_dark',
)
fig2.add_annotation(
    text=f'{cluster_count} coordinated groups detected',
    x=0.02,
    y=0.98,
    xref='paper',
    yref='paper',
    showarrow=False,
    bgcolor='rgba(0,0,0,0.4)',
)
fig2.show()

In [5]:
cluster_stats = (
    analysis_df[analysis_df['cluster_id'] >= 0]
    .groupby('cluster_id', as_index=False)
    .agg(wallets=('address', 'count'), mean_sybil_probability=('sybil_probability', 'mean'))
    .sort_values('wallets', ascending=False)
)

fig3 = px.bar(
    cluster_stats,
    x='cluster_id',
    y='wallets',
    color='mean_sybil_probability',
    color_continuous_scale='RdYlGn_r',
    title='Chart 3. Cluster Size Distribution',
    labels={
        'cluster_id': 'Cluster ID',
        'wallets': 'Number of wallets',
        'mean_sybil_probability': 'Mean sybil probability',
    },
    template='plotly_dark',
)
largest_cluster = int(cluster_stats.iloc[0]['cluster_id'])
largest_wallets = int(cluster_stats.iloc[0]['wallets'])
fig3.add_annotation(
    text=f'Largest cluster: {largest_cluster} ({largest_wallets} wallets)',
    x=largest_cluster,
    y=largest_wallets,
    showarrow=True,
    arrowhead=2,
)
fig3.show()

## 3) Inside a Sybil Cluster

We now zoom into the largest detected cluster and inspect coordination evidence. We focus on timing synchrony, gas strategy similarity, and upstream funding patterns. This mirrors how an analyst would build a removal list before allocation.

In [6]:
largest_cluster_df = analysis_df[analysis_df['cluster_id'] == largest_cluster].copy()
cluster_addresses = largest_cluster_df['address'].tolist()

coord_breakdown_df = pd.DataFrame(largest_cluster_df['coordination_breakdown'].tolist()).fillna(0.0)
coord_score = float(coord_breakdown_df.get('coordination_score', pd.Series([0.0])).mean())

print(f"Let's investigate Cluster {largest_cluster} — {len(cluster_addresses)} wallets with coordination score {coord_score:.3f}.")

Let's investigate Cluster 0 — 15 wallets with coordination score 0.722.


In [7]:
cluster_tx = tx[tx['address'].isin(cluster_addresses)].copy()
cluster_tx['datetime'] = pd.to_datetime(cluster_tx['timestamp'], unit='s', utc=True)

cluster_tx['tx_type'] = np.select(
    [
        cluster_tx['from_addr'] == cluster_tx['address'],
        cluster_tx['to_addr'] == cluster_tx['address'],
    ],
    ['outgoing', 'incoming'],
    default='other',
)

fig4 = px.scatter(
    cluster_tx.sort_values('datetime'),
    x='datetime',
    y='address',
    color='tx_type',
    title=f'Chart 4. Cluster {largest_cluster} Timeline of Coordinated Actions',
    labels={
        'datetime': 'Transaction timestamp (UTC)',
        'address': 'Wallet address',
        'tx_type': 'Transaction type',
    },
    template='plotly_dark',
)
fig4.add_annotation(
    text='Wallets repeatedly act in tight windows across multiple days',
    x=0.01,
    y=1.08,
    xref='paper',
    yref='paper',
    showarrow=False,
)
fig4.show()

In [8]:
fig5 = px.box(
    cluster_tx,
    x='address',
    y='gas_price',
    points='all',
    title=f'Chart 5. Gas Pattern Fingerprint for Cluster {largest_cluster}',
    labels={'address': 'Wallet address', 'gas_price': 'Gas price (wei)'},
    template='plotly_dark',
)
cluster_median_gas = float(cluster_tx['gas_price'].median())
fig5.add_hline(
    y=cluster_median_gas,
    line_dash='dash',
    line_color='cyan',
    annotation_text=f'Cluster median gas: {cluster_median_gas:,.0f}',
)
fig5.update_xaxes(showticklabels=False)
fig5.show()

cluster_median_gas_cv = float(features[features['address'].isin(cluster_addresses)]['gas_price_cv'].median())
print(f'Near-identical gas strategy across the cluster — median gas CV = {cluster_median_gas_cv:.4f}.')

Near-identical gas strategy across the cluster — median gas CV = 0.0847.


In [9]:
incoming_funding = cluster_tx[(cluster_tx['to_addr'].isin(cluster_addresses)) & (~cluster_tx['from_addr'].isin(cluster_addresses))].copy()
edge_df = (
    incoming_funding.groupby(['from_addr', 'to_addr'], as_index=False)
    .size()
    .rename(columns={'size': 'edge_weight'})
)

if edge_df.empty:
    edge_df = pd.DataFrame({'from_addr': ['no_external_funder'], 'to_addr': [cluster_addresses[0]], 'edge_weight': [1]})

funders = edge_df['from_addr'].drop_duplicates().tolist()
wallets = sorted(edge_df['to_addr'].drop_duplicates().tolist())

node_x = {}
node_y = {}
for i, f in enumerate(funders):
    node_x[f] = 0.0
    node_y[f] = i - (len(funders) - 1) / 2
for j, w in enumerate(wallets):
    node_x[w] = 1.0
    node_y[w] = j - (len(wallets) - 1) / 2

line_traces = []
for _, r in edge_df.iterrows():
    line_traces.append(
        go.Scatter(
            x=[node_x[r['from_addr']], node_x[r['to_addr']]],
            y=[node_y[r['from_addr']], node_y[r['to_addr']]],
            mode='lines',
            line=dict(width=1 + float(r['edge_weight']), color='rgba(255,140,0,0.55)'),
            hoverinfo='text',
            text=f"{r['from_addr'][:10]} -> {r['to_addr'][:10]} | events={int(r['edge_weight'])}",
            showlegend=False,
        )
    )

node_df = pd.DataFrame(
    {
        'node': funders + wallets,
        'x': [node_x[n] for n in funders + wallets],
        'y': [node_y[n] for n in funders + wallets],
        'type': ['funder'] * len(funders) + ['cluster_wallet'] * len(wallets),
    }
)

fig6 = go.Figure()
for tr in line_traces:
    fig6.add_trace(tr)
fig6.add_trace(
    go.Scatter(
        x=node_df['x'],
        y=node_df['y'],
        mode='markers+text',
        marker=dict(
            size=np.where(node_df['type'] == 'funder', 16, 10),
            color=np.where(node_df['type'] == 'funder', '#ff6b6b', '#4dabf7'),
            line=dict(color='white', width=1),
        ),
        text=[n[:10] + '…' for n in node_df['node']],
        textposition='top center',
        hovertext=node_df['node'],
        hoverinfo='text',
        showlegend=False,
    )
)
fig6.update_layout(
    title=f'Chart 6. Funding Flow into Cluster {largest_cluster}',
    template='plotly_dark',
    xaxis_title='Funding direction (upstream funder → cluster wallet)',
    yaxis_title='Wallet index',
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
)
fig6.add_annotation(
    text='Shared upstream funding creates strong linkage evidence',
    x=0.5,
    y=1.06,
    xref='paper',
    yref='paper',
    showarrow=False,
)
fig6.show()

if not incoming_funding.empty:
    top_funder = incoming_funding['from_addr'].value_counts().idxmax()
    fspan_min = (incoming_funding['timestamp'].max() - incoming_funding['timestamp'].min()) / 60.0
    print(f'All {len(wallets)} wallets are funded from {top_funder} within {fspan_min:.1f} minutes.')

All 15 wallets are funded from 0x0000000000000000000000000000000000f00000 within 24690.9 minutes.


## 4) The Explainer — Why These Wallets Are Linked

We now convert cluster signals into plain-English evidence. This gives risk teams a short narrative they can attach to governance or distribution decisions.

In [10]:
explanation = explain_wallet_linkage(cluster_addresses, tx, features)

radar_scores = {
    'Timing': float(coord_breakdown_df.get('temporal_correlation', pd.Series([0.0])).mean()),
    'Gas': float(coord_breakdown_df.get('gas_similarity', pd.Series([0.0])).mean()),
    'Funding': float(coord_breakdown_df.get('common_funding', pd.Series([0.0])).mean()),
    'Sequencing': float(coord_breakdown_df.get('sequential_activity', pd.Series([0.0])).mean()),
}

fig7 = go.Figure(
    data=go.Scatterpolar(
        r=list(radar_scores.values()),
        theta=list(radar_scores.keys()),
        fill='toself',
        name='Evidence strength',
        marker=dict(color='#ff6b6b'),
    )
)
fig7.update_layout(
    template='plotly_dark',
    title='Chart 7. Evidence Breakdown for the Selected Cluster',
    polar=dict(radialaxis=dict(visible=True, range=[0, 1], title='Score (0-1)')),
)
fig7.show()

print('Plain-English explainer output:')
for idx, sentence in enumerate(explanation.get('evidence_sentences', []), start=1):
    print(f'{idx}. {sentence}')
print('Confidence score:', round(float(explanation.get('confidence_score', 0.0)), 3))

Plain-English explainer output:
1. Transact in same minute-buckets across 8 windows.
2. Near-identical gas strategy (median gas_price CV=0.008, gas_used CV=0.027).
3. Recurring contract interaction sequence similarity: 0.97.
Confidence score: 0.62


## 5) Airdrop Hunter Mode

We run campaign-level scoring over all participant wallets to estimate abuse rate and prioritize manual review. This mode gives both cluster-level and wallet-level decisions. The output is designed for an airdrop eligibility pipeline.

In [11]:
campaign_results = run_airdrop_hunter(
    participant_addresses=participants,
    transactions=tx,
    chain='ethereum',
)

summary = campaign_results['campaign_summary']
wallet_scores_df = pd.DataFrame(campaign_results['wallet_scores'])
marginal_count = len(campaign_results.get('marginal_members', []))
confirmed_count = int(summary.get('estimated_sybil_participants', 0))
participant_count = int(campaign_results.get('participant_count', len(participants)))
legit_count = max(participant_count - confirmed_count - marginal_count, 0)

abuse_rate = 100.0 * (confirmed_count + marginal_count) / max(participant_count, 1)
print(
    f"Campaign abuse summary: {abuse_rate:.2f}% estimated Sybil rate, "
    f"{summary.get('cluster_count', 0)} farmer clusters identified."
)

fig8 = go.Figure(
    data=[
        go.Pie(
            labels=['Legitimate', 'Suspected Sybil', 'Confirmed Sybil'],
            values=[legit_count, marginal_count, confirmed_count],
            hole=0.55,
            marker=dict(colors=['#2ca02c', '#ff7f0e', '#d62728']),
            textinfo='label+percent',
        )
    ]
)
fig8.update_layout(
    template='plotly_dark',
    title='Chart 8. Campaign Health Donut — Participant Risk Mix',
)
fig8.add_annotation(text=f'Estimated abuse:\n{abuse_rate:.1f}%', x=0.5, y=0.5, showarrow=False, font=dict(size=16))
fig8.show()

Campaign abuse summary: 22.71% estimated Sybil rate, 10 farmer clusters identified.


In [12]:
top20 = wallet_scores_df.sort_values('wallet_suspicion_score', ascending=False).head(20).copy()
top20['wallet_short'] = top20['address'].str.slice(0, 10) + '...'

fig9 = px.bar(
    top20.sort_values('wallet_suspicion_score', ascending=True),
    x='wallet_suspicion_score',
    y='wallet_short',
    orientation='h',
    color='wallet_suspicion_score',
    color_continuous_scale='RdYlGn_r',
    title='Chart 9. Top 20 Most Suspicious Wallets',
    labels={
        'wallet_suspicion_score': 'Suspicion score',
        'wallet_short': 'Wallet (truncated)',
    },
    template='plotly_dark',
)
fig9.add_annotation(
    text='These wallets should be excluded or manually reviewed before distribution',
    x=0.5,
    y=1.06,
    xref='paper',
    yref='paper',
    showarrow=False,
)
fig9.show()

## 6) How Resilient Is Our Detection?

Farmers adapt quickly, so we test across 8 adversarial difficulty levels. This stress test shows where the detector is robust and where we still need stronger signals.

In [13]:
adv_rows = []
for level in range(1, 9):
    tx_adv, labels_adv = generate_adversarial_sybils(
        difficulty=level,
        num_clusters=8,
        wallets_per_cluster=10,
        seed=SEED,
    )
    feats_adv = extract_features(tx_adv)
    preds_adv = SybilDetector().fit_predict(feats_adv)
    merged_adv = preds_adv.merge(labels_adv[['address', 'is_sybil']], on='address', how='inner')

    y_true = merged_adv['is_sybil'].astype(int)
    y_pred = (merged_adv['sybil_probability'] >= 0.5).astype(int)

    adv_rows.append(
        {
            'Level': level,
            'Precision': float(precision_score(y_true, y_pred, zero_division=0)),
            'Recall': float(recall_score(y_true, y_pred, zero_division=0)),
            'F1': float(f1_score(y_true, y_pred, zero_division=0)),
        }
    )

adv_df = pd.DataFrame(adv_rows)
heat_df = adv_df.set_index('Level')[['Precision', 'Recall', 'F1']]

fig10 = px.imshow(
    heat_df,
    text_auto='.3f',
    color_continuous_scale='RdYlGn',
    zmin=0,
    zmax=1,
    title='Chart 10. Adversarial Resilience Heatmap (Levels 1-8)',
    labels={'x': 'Metric', 'y': 'Difficulty level', 'color': 'Score'},
    template='plotly_dark',
)
fig10.update_xaxes(title='Metric')
fig10.update_yaxes(title='Difficulty level')
fig10.show()

hardest_row = adv_df.sort_values('F1').iloc[0]
print(
    f"Detection remains strong through most levels. Hardest case: Level {int(hardest_row['Level'])} "
    f"with F1={hardest_row['F1']:.3f}."
)
print('F1 range:', f"{adv_df['F1'].min():.3f} to {adv_df['F1'].max():.3f}")

Detection remains strong through most levels. Hardest case: Level 3 with F1=0.857.
F1 range: 0.857 to 1.000


## 7) Multi-Chain Ready

The same detection pipeline works across all major EVM chains with one config change.

In [14]:
chain_id_map = {
    'ethereum': 1,
    'base': 8453,
    'bnb': 56,
    'arbitrum': 42161,
    'optimism': 10,
    'polygon': 137,
}
chain_df = pd.DataFrame(
    {'chain': SUPPORTED_CHAINS, 'chain_id': [chain_id_map[c] for c in SUPPORTED_CHAINS]}
)

fig11 = px.bar(
    chain_df,
    x='chain',
    y='chain_id',
    text='chain_id',
    title='Chart 11. Supported EVM Chains and Chain IDs',
    labels={'chain': 'Chain', 'chain_id': 'Chain ID'},
    template='plotly_dark',
    color='chain_id',
    color_continuous_scale='Blues',
)
fig11.update_traces(textposition='outside')
fig11.show()

print('The same detector can scan Ethereum, Base, BNB, Arbitrum, Optimism, and Polygon.')

The same detector can scan Ethereum, Base, BNB, Arbitrum, Optimism, and Polygon.


Built with [onchain-sybil-detector](https://github.com/KOKOSde/onchain-sybil-detector) — open-source Sybil detection for DeFi, airdrops, and crypto exchanges.